In [ ]:
# jupyter nbconvert "data/<your_input_file>" --no-input --to html

# Deep sequencing analysis for GS BTHC Batch1 part2

# QC module

## 1 Functions and module

### 1.1 Modules

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
import numpy as np
import seaborn as sns
import scipy

In [ ]:
pd.set_option('display.max_columns', 30)

### 1.2 Functions and Constants

In [ ]:
# Column name constants
TOTAL_TUMOR_NUMBER = 'TTN'  # Total Tumor Number
TOTAL_TUMOR_BURDEN = 'TTB'  # Total Tumor Burden


In [ ]:
def filter_and_aggregate(df, n, aggregating_column='Targeted_gene_name'):
    """
    For each Sample_ID:
      - Select the top `n` rows ranked by total tumor burden.
      - Aggregate all remaining rows into a single "others" entry.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with sample data
    n : int
        Number of top rows to keep per sample
    aggregating_column : str, default 'Targeted_gene_name'
        Column name to use for aggregation (e.g., 'Targeted_gene_name' or 'gRNA')
    
    Returns
    -------
    pd.DataFrame
        DataFrame with top n rows per sample + aggregated 'others' entry
    """
    aggregated_results = []

    for sample_id, group in df.groupby('Sample_ID'):
        # Top n by total tumor burden
        top_rows = group.nlargest(n, TOTAL_TUMOR_BURDEN)

        # Aggregate remaining rows into "others"
        if len(group) > n:
            remaining = group.iloc[n:]

            # Build others row with all non-grouping columns preserved
            others_row_dict = {
                'Sample_ID': [sample_id],
                aggregating_column: ['others'],
                TOTAL_TUMOR_NUMBER: [remaining[TOTAL_TUMOR_NUMBER].sum()],
                TOTAL_TUMOR_BURDEN: [remaining[TOTAL_TUMOR_BURDEN].sum()],
            }
            
            # Preserve other columns from the original dataframe (e.g., Mouse_genotype, Tissue_type)
            for col in group.columns:
                if col not in ['Sample_ID', aggregating_column, TOTAL_TUMOR_NUMBER, TOTAL_TUMOR_BURDEN]:
                    # Use the first value (should be consistent within sample)
                    others_row_dict[col] = [group[col].iloc[0]]

            others_row = pd.DataFrame(others_row_dict)

            top_rows = pd.concat([top_rows, others_row], ignore_index=True)

        aggregated_results.append(top_rows)

    # Combine all samples together
    return pd.concat(aggregated_results, ignore_index=True)

In [ ]:
def label_point(x, y, val, ax):
    a = pd.concat({'x': x, 'y': y, 'val': val}, axis=1)
    for i, point in a.iterrows():
        ax.text(point['x']+.02, point['y'], str(point['val']),size = 8)

In [ ]:
def generate_sample_summary(input_df,input_cell_number_cutoff,input_read_cutoff):
    # input_cell_number_cutoff is the cell number cutoff
    # input_rad_cutoff is the read cutoff
    # Total reads does not restricted to gRNA or cell number cutoff
    temp_df0 = input_df.groupby(['Sample_ID','Mouse_Ear_Tag','Mouse_genotype', 'Sex','Pooling_library_name',
         'Time_after_tumor_initiation', 'Total_lung_weight', 'Virus_titer','Correction_for_spikein','Cell_number_per_read','Tissue_type'],as_index=False).agg(
        TTR = pd.NamedAgg('Count',aggfunc = sum))
    # Calculate spike-in read ratio for each sample using 'Count' (spike-in / TTR)
    spikein_counts = input_df[input_df['Identity'] == 'Spikein'].groupby('Sample_ID')['Count'].sum()
    temp_df0['Spikein_read_ratio'] = temp_df0['Sample_ID'].map(spikein_counts).fillna(0) / temp_df0['TTR']
    # filter input data
    temp_input = input_df[(input_df['Cell_number']>=input_cell_number_cutoff)&(input_df['Count']>input_read_cutoff)]
    # I only consider non-spikein gRNA
    temp_df1 = temp_input[temp_input['Identity']=='gRNA'].groupby(
        ['Sample_ID'],as_index = False).apply(
        cal_sample_summary)
    temp_df1['TTB_million'] = temp_df1['TTB']/1000000
    # merge sample and gRNA information    
    temp_df1 = temp_df1.merge(temp_df0,on = 'Sample_ID',how = 'right')
    # normalize to per 100K virus
    temp_df1['Tumor number per 100K virus'] = temp_df1.apply(lambda x: x['TTN']/x['Virus_titer']*100000,axis=1)
    temp_df1['Total tumor burden (million per 100K virus)'] = np.log10(temp_df1.apply(lambda x: x['TTB_million']/x['Virus_titer']*100000,axis=1))
    weight_per_cell_ng = ((temp_df1['Total_lung_weight'] - 0.15) * 1e9) / temp_df1['TTB']
    weight_per_cell_ng = weight_per_cell_ng.where((weight_per_cell_ng > 0) & np.isfinite(weight_per_cell_ng), np.nan)
    temp_df1['Weight_per_cell_ng'] = weight_per_cell_ng
    return(temp_df1)
# calculate the summary metrics for each sample
def cal_sample_summary(x):
    d = {}
    temp_vect = x['Cell_number']
    if type (temp_vect) == 'int':
        temp_vect = [temp_vect]
    d['gRNA_recovered'] = len(x['gRNA'].unique())
    d['TTB'] = sum(temp_vect) # total mutational burdern 
    d['TTN'] = len(temp_vect) # this is total tumor number
    return pd.Series(d, index=list(d.keys())) 

----

## 2 Input and output address

In [ ]:
parent_address = "data/"
project_prefix = 'GS_BTHC_Batch1'
sample_summary_address = parent_address + f"{project_prefix}_sample_summary_df.csv"
annotated_data_output_address = parent_address + f"{project_prefix}_annotated_df.parquet"
fig_output_address1 =  f"figs/{project_prefix}_QC_figv1.pdf"
fig_output_address2 =  f"figs/{project_prefix}_QC_figv2.pdf"
final_data_output_address = parent_address + f"{project_prefix}_final_df.parquet"

-----

## 3 Data input and preprocessing

### 3.1 Read data

In [ ]:
annotated_df  = pd.read_parquet(annotated_data_output_address)

---

## 4 Top N genes and tumors

### 4.1 TTB burden seperation based on gene

In [ ]:
# Configuration constants
AGGREGATING_COLUMN = 'Targeted_gene_name'  # Can be changed to 'gRNA' if I want to aggregate on gRNA level
TOP_N_GENES = 5

# Aggregate genotype data: total tumor number and total tumor burden
# Filter to samples with >200 cells for statistical robustness
genotype_summary = (
    annotated_df[annotated_df.Cell_number > 200]
    .groupby(['Sample_ID', AGGREGATING_COLUMN, 'Mouse_genotype', 'Tissue_type'], as_index=False)
    .agg(
        **{TOTAL_TUMOR_NUMBER: pd.NamedAgg('Clonal_barcode', aggfunc='count'),
           TOTAL_TUMOR_BURDEN: pd.NamedAgg('Cell_number', aggfunc='sum')}
    )
)

# Exclude spike-in controls from analysis
non_spikein_summary = genotype_summary[
    ~genotype_summary[AGGREGATING_COLUMN].str.contains('Spike', case=False, na=False)
]

# Filter to top N units per sample and aggregate remaining as 'others'
filtered_summary = filter_and_aggregate(non_spikein_summary, TOP_N_GENES, aggregating_column=AGGREGATING_COLUMN)

In [ ]:
temp_query = filtered_summary
sample_ids = temp_query['Sample_ID'].unique()
unit_names = temp_query[AGGREGATING_COLUMN].unique()
num_samples = len(sample_ids)

palette = sns.color_palette("tab20", len(unit_names))
color_map = {unit: color for unit, color in zip(unit_names, palette)}

# --- SECTION 2: Define subplot grid (6 columns per row, stacked bar for each sample) ---

num_cols = 6
num_rows = -(-num_samples // num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 4.5, num_rows * 4))
axes = axes.flatten()

# --- SECTION 3: Plot stacked bar chart for each sample ---

for i, sample_id in enumerate(sample_ids):
    temp_df_sample = temp_query[temp_query['Sample_ID'] == sample_id]
    mouse_genotype = temp_df_sample['Mouse_genotype'].values[0]
    tissue = temp_df_sample['Tissue_type'].values[0]
    total_TTB = temp_df_sample[TOTAL_TUMOR_BURDEN].sum()
    sorted_df = temp_df_sample.sort_values(by=TOTAL_TUMOR_BURDEN, ascending=False)

    ax = axes[i]
    bottom = 0
    local_handles = []
    local_labels = []

    for _, row in sorted_df.iterrows():
        unit = row[AGGREGATING_COLUMN]
        height = row[TOTAL_TUMOR_BURDEN]
        percentage = height / total_TTB * 100

        if percentage < 0.1:
            continue  # skip small contributions

        bar = ax.bar(sample_id, height, bottom=bottom, color=color_map[unit])
        bottom += height

        # Collect handle/label for local legend
        local_handles.append(bar)
        ttn = row[TOTAL_TUMOR_NUMBER]
        local_labels.append(f"{unit} (TTN={ttn})")

    ax.set_title(f'{sample_id}: {mouse_genotype}: {tissue}')
    ax.set_ylabel(TOTAL_TUMOR_BURDEN)
    ax.set_xticks([])

    if local_handles:
        legend_title = "Gene combinations" if AGGREGATING_COLUMN == 'Targeted_gene_name' else "gRNA combinations"
        ax.legend(
            [h[0] for h in local_handles],
            local_labels,
            title=legend_title,
            loc='upper right',
            fontsize=6,
            title_fontsize=7,
            handlelength=1.2,
            borderpad=0.3,
            labelspacing=0.3
        )

# --- SECTION 4: Clean up unused subplots ---

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# --- SECTION 5: Final layout ---

plt.tight_layout()
plt.show()
fig.savefig("figs/TTB_stacked_bars_per_sample.pdf", format="pdf", bbox_inches="tight")

### 4.2 TTB burden seperation based on tumor

In [ ]:
# Configuration constants
TOP_N_ROWS = 10
MIN_CELL_NUMBER = 200

# Select columns to keep
selected_columns = [
    'Sample_ID', 'Targeted_gene_name', 'gRNA', 'Clonal_barcode',
    'Mouse_genotype', 'Cell_number', 'Total_lung_weight', 'Tissue_type'
]

# Filter: exclude spike-ins and samples with low cell counts
filtered_df = annotated_df[
    (~annotated_df['Targeted_gene_name'].str.contains('Spike', case=False, na=False)) &
    (annotated_df.Cell_number >= MIN_CELL_NUMBER)
][selected_columns]

# For each sample, keep top n rows by cell number and aggregate remaining as "others"
result_list = []
for sample_id, group in filtered_df.groupby('Sample_ID'):
    # Sort by cell number and keep top n rows
    top_rows = group.nlargest(TOP_N_ROWS, 'Cell_number')

    # Aggregate the remaining rows as "others"
    if len(group) > TOP_N_ROWS:
        remaining = group.iloc[TOP_N_ROWS:].copy()
        others_row = pd.DataFrame({
            'Sample_ID': [sample_id],
            'Targeted_gene_name': ['others'],
            'gRNA': ['others'],
            'Clonal_barcode': ['others'],
            'Total_lung_weight': [group['Total_lung_weight'].iloc[0]],
            'Mouse_genotype': [group['Mouse_genotype'].iloc[0]],  # Consistent within sample
            'Cell_number': [remaining['Cell_number'].sum()],
            'Tissue_type': [group['Tissue_type'].iloc[0]],
        })
        top_rows = pd.concat([top_rows, others_row], ignore_index=True)

    result_list.append(top_rows)

# Concatenate results into final DataFrame
tumor_df = pd.concat(result_list, ignore_index=True)

In [ ]:
# Configuration constants for plotting
PLOT_COLS_PER_ROW = 6
MIN_PERCENTAGE_THRESHOLD = 0.1  # Minimum percentage to display in stacked bars
PLOT_FIGURE_WIDTH = 4.5
PLOT_FIGURE_HEIGHT = 4

# Prepare data for plotting
plot_df = tumor_df
sample_ids = plot_df['Sample_ID'].unique()
unit_names = plot_df[AGGREGATING_COLUMN].unique()
num_samples = len(sample_ids)

# Create color palette and mapping
palette = sns.color_palette("tab20", len(unit_names))
color_map = {unit: color for unit, color in zip(unit_names, palette)}

# Define subplot grid (6 columns per row, 2 plots per sample)
num_cols = PLOT_COLS_PER_ROW
num_rows = -(-num_samples * 2 // num_cols)

# Create subplots
fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * PLOT_FIGURE_WIDTH, num_rows * PLOT_FIGURE_HEIGHT))
axes = axes.flatten()

# Plot loop: create two stacked bar charts per sample
for i, sample_id in enumerate(sample_ids):
    sample_data = plot_df[plot_df['Sample_ID'] == sample_id]
    mouse_genotype = sample_data['Mouse_genotype'].values[0]
    lung_weight = sample_data['Total_lung_weight'].values[0]
    total_tumor_burden = sample_data['Cell_number'].sum()

    # --- First Stacked Bar: All combinations (including "others") ---
    ax1 = axes[i * 2]
    sorted_df = sample_data.sort_values(by='Cell_number', ascending=False)
    bottom = 0
    
    for _, row in sorted_df.iterrows():
        height = row['Cell_number']
        percentage = height / total_tumor_burden * 100
        
        if percentage < MIN_PERCENTAGE_THRESHOLD:
            continue
        
        unit = row[AGGREGATING_COLUMN]
        ax1.bar(sample_id, height, bottom=bottom, color=color_map[unit])
        bottom += height

    ax1.set_title(
        f'Original: {sample_id}, {mouse_genotype}\n'
        f'Tumor burden={total_tumor_burden:.2e}, Lung weight={lung_weight:.2f}'
    )
    ax1.set_ylabel('Cell Number')
    ax1.set_xticks([])

    # --- Second Stacked Bar: Filtered (excluding "others") + Local Legend ---
    ax2 = axes[i * 2 + 1]
    filtered_data = sample_data[sample_data[AGGREGATING_COLUMN] != 'others']
    total_tumor_burden_filtered = filtered_data['Cell_number'].sum()
    bottom = 0
    local_handles = []
    local_labels = []

    if not filtered_data.empty:
        sorted_df_filtered = filtered_data.sort_values(by='Cell_number', ascending=False)
        
        for _, row in sorted_df_filtered.iterrows():
            height = row['Cell_number']
            percentage = height / total_tumor_burden_filtered * 100
            
            if percentage < MIN_PERCENTAGE_THRESHOLD:
                continue
            
            unit = row[AGGREGATING_COLUMN]
            bar = ax2.bar(sample_id, height, bottom=bottom, color=color_map[unit])
            bottom += height
            
            # Collect for legend
            local_handles.append(bar)
            local_labels.append(unit)

        ax2.set_title(
            f'Top Tumors: {sample_id}, {mouse_genotype}\n'
            f'Tumor burden={total_tumor_burden_filtered:.2e}'
        )
        ax2.set_ylabel('Cell Number')
        ax2.set_xticks([])

        # Add local legend with dynamic title
        legend_title = "Gene combinations" if AGGREGATING_COLUMN == 'Targeted_gene_name' else "gRNA combinations"
        ax2.legend(
            [h[0] for h in local_handles],
            local_labels,
            title=legend_title,
            loc='upper right',
            fontsize=6,
            title_fontsize=7,
            handlelength=1.2,
            borderpad=0.3,
            labelspacing=0.3
        )

# Hide unused subplots
for j in range(i * 2 + 2, len(axes)):
    fig.delaxes(axes[j])

# Final layout adjustment
plt.tight_layout()
plt.show()
fig.savefig("figs/Top10_tumor_stacked_bars_with_local_legends.pdf", format="pdf", bbox_inches="tight")

## 5 Overall sample QC 

### 5.1 Explaination

In [ ]:
cell_number_cutoff = 100
Sample_summary_df =  generate_sample_summary(annotated_df,cell_number_cutoff,2)

In [ ]:
Sample_summary_df['R_L_ratio'] = Sample_summary_df.TTR/(Sample_summary_df.Total_lung_weight-0.15)

In [ ]:
Sample_summary_df.head()

### 5.2 Plot v1

* <font size="5" color =  red> The current plot is slightly different different from the conventional UltraSeq plot because I plot dat from different tissues as well. </font>
* <font size="5" color =  red> For panel A, I just want to show the how many cells each read represents</font>
* <font size="5" color =  blue> For panel B, most of the dots show positive correlation between tumor burden and tumor weight.</font>
* <font size="5" color =  blue> For panel B, dots deviated are mostly dots with little reads mapped, which has a smaller dot.</font>
* <font size="5" color =  green> For panel C, same as panel B. Here I just want to check if any spike in corrected sample will deviate from the line. Because if the spike-in reads are wrong, they will not follow the general trend</font>
* <font size="5" color =  brown> For panel D, I want to show the spikein ratio and total number of reads mapped to each sample. We expect a negative correlation. I labled reads with very low reads mapped (<0.1M) </font>
* <font size="5" color =  orange> For panel E, I want to show the number of gRNA recovered. I labeled samples with less than 98% sgRNA recovered</font>
* <font size="5" color =  purple> For panel F, I want to check if there is saturation of tumor burden with the increasing of more tumor. It seems there isn't</font>
* <font size="5" color =  oliva> For panel G, I want to show the relation between virus titer and TTB. Male seems to have higher tumor burden regardless of the genotype. KTC and KTHC seems to have similar burden</font>
* <font size="5" color =  oliva> For panel H, I want to show the relation between virus titer and TTN.</font>
* <font size="5" color =  oliva> For panel I, similar to panel G.</font>

In [ ]:

gs = gridspec.GridSpec(3, 17) 
fig1 = plt.figure(figsize=(17,15))
# Optional: apply a clean seaborn style
# sns.set(style="whitegrid")
ax1=fig1.add_subplot(gs[:1, 0:5])
temp_df = Sample_summary_df
sns.ecdfplot(data=temp_df, x="Cell_number_per_read",ax = ax1)
ax1.set_xlabel('Cells number per read')
ax1.set_xscale('log')
ax1.axvline(50,color='red', linestyle='--')
ax1.axvline(100,color='black', linestyle='--')
ax1.axvline(1000,color='green', linestyle='--')
ax1.set_title('A', loc ='left')


ax2=fig1.add_subplot(gs[:1, 6:11])
temp_df = Sample_summary_df[(Sample_summary_df['Total_lung_weight']!= 0)&(~Sample_summary_df['TTB_million'].isna())]
sns.regplot(x='Total_lung_weight', y='TTB_million', data=temp_df, fit_reg=True ,ax= ax2,scatter=False,color = 'tab:grey')
sns.scatterplot(x='Total_lung_weight', y='TTB_million', data=temp_df, hue = 'Mouse_genotype',ax = ax2,size= 'TTR')
# label_point(temp_df['Total_lung_weight'], temp_df['TTB_million'], temp_df['Sample_ID'], plt.gca()) # this is for labeling
ax2.set_xlabel('Total lung weight')
ax2.set_ylabel('Total tumor burden (million cells)')
ax2.set_title('B', loc ='Left')

ax3=fig1.add_subplot(gs[:1, 12:17])
sns.regplot(x='Total_lung_weight', y='TTB_million', data=temp_df, fit_reg=True ,ax= ax3,scatter=False,color = 'tab:grey')
sns.scatterplot(x='Total_lung_weight', y='TTB_million', data=temp_df, ax = ax3)
label_point(temp_df['Total_lung_weight'], temp_df['TTB_million'], temp_df['Sample_ID'], plt.gca()) # this is for labeling
ax3.set_xlabel('Total lung weight')
ax3.set_ylabel('Total tumor burden (million cells)')
ax3.set_title('C', loc ='Left')


ax4 = fig1.add_subplot(gs[1:2, 0:5])
sns.scatterplot(x='TTR', y='Spikein_read_ratio', data=Sample_summary_df, hue='Mouse_genotype', ax= ax4)
temp_df = Sample_summary_df[Sample_summary_df.TTR<100000] # < 0.1 million reads
label_point(temp_df['TTR'], temp_df['Spikein_read_ratio'], temp_df['Sample_ID'], plt.gca()) # this is for labeling
# ax4.set_yscale('log')
ax4.set_title('D', loc ='Left')
ax4.set_xlabel('Total reads mapped')
ax4.set_ylabel('Spike in reads ratio')

ax5 = fig1.add_subplot(gs[1:2, 6:11])
sns.scatterplot(x='TTR', y='gRNA_recovered',data=Sample_summary_df,hue='Mouse_genotype', ax= ax5)
temp_df = Sample_summary_df[Sample_summary_df.gRNA_recovered<len(annotated_df.gRNA.unique())*0.9] # < 90 gRNA recovered
label_point(temp_df['TTR'], temp_df['gRNA_recovered'], temp_df['Sample_ID'], plt.gca()) # this is for labeling
ax5.set_xlabel('Total reads mapped')
ax5.set_ylabel('gRNA recovered')
ax5.set_xscale('log', base=10)
ax5.set_title('E', loc ='left')


temp_df = Sample_summary_df[Sample_summary_df.TTR>1000000] # I request at least 1 million reads
ax6 = fig1.add_subplot(gs[1:2, 12:17])
sns.regplot(x='TTN', y='TTB', data=temp_df, fit_reg=True ,ax= ax6,scatter=False,color = 'tab:grey')
sns.scatterplot(x='TTN', y='TTB', data=temp_df, hue='Mouse_genotype', ax= ax6)
# ax6.set_yscale('log')
ax6.set_title('F', loc ='Right')
ax6.set_xlabel('TTN')
ax6.set_ylabel('TTB')
# ax6.xaxis.major.formatter._useMathText = True
ax6.ticklabel_format(axis='both',style='sci',scilimits=(-3,4))
ax6.legend(loc='upper left')


ax7 = fig1.add_subplot(gs[2:3, :5])
# sns.boxplot(x='Virus_titer', y='TTB_million', data=Sample_summary_df, hue=Sample_summary_df[['Mouse_genotype','Sex']].apply(tuple, axis=1), ax= ax7)
sns.boxplot(x='Tissue_type', y='TTB_million', data=Sample_summary_df, hue='Mouse_genotype', ax= ax7)
ax7.set_title('G', loc ='right')
ax7.set_ylabel('Total tumor burden (million cells)')

ax8 = fig1.add_subplot(gs[2:3, 6:11])
# sns.boxplot(x='Virus_titer', y='TTN', data=Sample_summary_df, hue=Sample_summary_df[['Mouse_genotype','Sex']].apply(tuple, axis=1), ax= ax8)
sns.boxplot(x='Tissue_type', y='Tumor number per 100K virus', data=Sample_summary_df, hue='Mouse_genotype', ax= ax8)
ax8.set_title('H', loc ='Left')
ax8.set_ylabel('Tumor number per 100K virus')

ax9 = fig1.add_subplot(gs[2:3, 12:17])
sns.scatterplot(x='R_L_ratio', y='Spikein_read_ratio', data=Sample_summary_df, hue='Mouse_genotype', ax= ax9)
ax9.set_title('I', loc ='right')
ax9.set_xlabel('Reads/lung weight ratio')
test_df = Sample_summary_df[Sample_summary_df.Spikein_read_ratio>0.004]
label_point(test_df['R_L_ratio'], test_df['Spikein_read_ratio'], test_df['Sample_ID'], plt.gca()) # this is for labeling
ax9.set_ylabel('Spike in ratio')
fig1.savefig(fig_output_address1)

### 5.3 Nomination sample to exclude

In [ ]:
Sample_summary_df.sort_values(by = 'TTN')

In [ ]:
f1 = Sample_summary_df[Sample_summary_df['Cell_number_per_read']>200]['Sample_ID'] # read depth
f2 = Sample_summary_df[Sample_summary_df['Spikein_read_ratio']>0.5]['Sample_ID']
f3 = Sample_summary_df[Sample_summary_df['TTR']<100000]['Sample_ID'] # total read depth 
f4 = Sample_summary_df[Sample_summary_df['TTN']<50]['Sample_ID'] # tumor numbers

In [ ]:
sample_to_throw = list(set(f1)|set(f2)|set(f3)|set(f4))

In [ ]:
sample_to_throw

<font size="10" color =  red> I will not exlcude anything </font>

### 5.4 Output summary

In [ ]:
Sample_summary_df.to_csv(sample_summary_address,index=False)

In [ ]:
Sample_summary_df.head()

---

## 6 Contamination analysis

### 6.1 Data processing

In [ ]:
MIN_CELL_NUMBER = 100
input_df = annotated_df[annotated_df.Identity=='gRNA']
input_df = input_df[~input_df.Sample_ID.isin(sample_to_throw)].copy()
input_df = input_df[input_df.Cell_number>=MIN_CELL_NUMBER]
input_df['gRNA_clonalbarcode'] = input_df['gRNA'] + '_' + input_df['Clonal_barcode']

In [ ]:
temp = input_df.groupby('Sample_ID')['Count'].count()
ref_dict = dict(zip(temp.index, temp.values))

In [ ]:
import pandas as pd

# Assuming `input_df` is the DataFrame with columns 'Sample_ID' and 'gRNA_clonalbarcode'

# Step 1: Group by Sample_ID and collect unique gRNA_clonalbarcode values for each Sample_ID
sample_barcodes = input_df.groupby('Sample_ID')['gRNA_clonalbarcode'].apply(set).to_dict()

# Step 2: Get all unique Sample_IDs
sample_ids = list(sample_barcodes.keys())

# Step 3: Initialize an empty DataFrame to store the shared fractions
shared_fraction_df = pd.DataFrame(index=sample_ids, columns=sample_ids)

# Step 4: Calculate the shared fraction of gRNA_clonalbarcode in row Sample_ID (i) with column Sample_ID (j)
for id1 in sample_ids:
    for id2 in sample_ids:
        if id1 == id2:
            shared_fraction_df.loc[id1, id2] = 1.0  # Full overlap with itself
        else:
            # Calculate intersection and the total unique gRNA_clonalbarcode count for Sample_ID id1
            intersection = sample_barcodes[id1].intersection(sample_barcodes[id2])
            shared_fraction = len(intersection) / len(sample_barcodes[id1]) if sample_barcodes[id1] else 0
            shared_fraction_df.loc[id1, id2] = shared_fraction

# Convert all values to numeric
shared_fraction_df = shared_fraction_df.apply(pd.to_numeric)
# Convert the DataFrame to long format
shared_fraction_long_df = shared_fraction_df.reset_index().melt(id_vars='index', var_name='Sample_ID_j', value_name='shared_fraction')
shared_fraction_long_df.rename(columns={'index': 'Sample_ID_i'}, inplace=True)

# Remove rows where Sample_ID_i is equal to Sample_ID_j
# shared_fraction_long_df = shared_fraction_long_df[shared_fraction_long_df['Sample_ID_i'] != shared_fraction_long_df['Sample_ID_j']]
shared_fraction_long_df['Tumor_number_i'] = shared_fraction_long_df.Sample_ID_i.apply(lambda x: ref_dict.get(x))
shared_fraction_long_df['Tumor_number_j'] = shared_fraction_long_df.Sample_ID_j.apply(lambda x: ref_dict.get(x))



In [ ]:
filtered_df = shared_fraction_long_df[shared_fraction_long_df["Sample_ID_i"] != shared_fraction_long_df["Sample_ID_j"]]
# Group by Sample_ID_i and Tumor_number_i, calculating the mean shared_fraction
fraction_result_df = filtered_df.groupby(["Sample_ID_i", "Tumor_number_i"])["shared_fraction"].mean().reset_index()

# Rename columns
fraction_result_df.columns = ["Sample_ID", "Tumor_number", "mean_shared_fraction"]

In [ ]:
filtered_df.shared_fraction.mean()

<font size="10" color =  red> Remember I have 2 pools, cross pool sharing are minimal</font>

### 7.2 Distribution of shared fraction

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create the figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Plot histogram using seaborn
sns.histplot(fraction_result_df["mean_shared_fraction"], bins=10, kde=True, ax=ax)

# Customize labels and title
ax.set_xlabel("Mean Shared Fraction")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Mean Shared Fraction")

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Sort the DataFrame by Tumor_number_i and Tumor_number_j
sorted_df = shared_fraction_long_df.sort_values(['Tumor_number_i', 'Tumor_number_j'])

# Create a pivot table for the heatmap, sorted by Sample_IDs
heatmap_data = sorted_df.pivot(index='Sample_ID_i', columns='Sample_ID_j', values='shared_fraction')

# Ensure rows and columns are sorted based on Tumor_number
heatmap_data = heatmap_data.loc[sorted_df['Sample_ID_i'].unique(), sorted_df['Sample_ID_j'].unique()]

# Extract Tumor_number_i and Tumor_number_j values in the order of sorted Sample_IDs
tumor_number_i = sorted_df.drop_duplicates('Sample_ID_i').set_index('Sample_ID_i')['Tumor_number_i']
tumor_number_j = sorted_df.drop_duplicates('Sample_ID_j').set_index('Sample_ID_j')['Tumor_number_j']

# Plot the main heatmap
fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(
    heatmap_data, cmap='YlGnBu', cbar_kws={'label': 'Shared Fraction'},
    ax=ax, xticklabels=True, yticklabels=True
)
# Create colorbars (ladders) for Tumor_number_i (y-axis) and Tumor_number_j (x-axis)
# Normalize the tumor numbers for consistent color mapping
norm_i = plt.Normalize(tumor_number_i.min(), tumor_number_i.max())
norm_j = plt.Normalize(tumor_number_j.min(), tumor_number_j.max())
cmap_i = plt.cm.Reds
cmap_j = plt.cm.Reds

# Adjust thickness for ladders (0.3 width for y-axis ladder, 0.3 height for x-axis ladder)
ladder_thickness_y = 0.2
ladder_thickness_x = 0.2

# Add the ladder for Tumor_number_i on the left side
for i, value in enumerate(tumor_number_i):
    ax.add_patch(plt.Rectangle((-ladder_thickness_y, i), ladder_thickness_y, 1, color=cmap_i(norm_i(value))))

# Add the ladder for Tumor_number_j on the top
for j, value in enumerate(tumor_number_j):
    ax.add_patch(plt.Rectangle((j, -ladder_thickness_x), 1, ladder_thickness_x, color=cmap_j(norm_j(value))))

# Adjust plot limits to make space for the x-axis ladder
ax.set_ylim(len(heatmap_data), -0.5)
ax.set_xlim(-0.5, len(tumor_number_j))

# Add a color bar for the x-axis ladder legend
sm = plt.cm.ScalarMappable(cmap=cmap_j, norm=norm_j)
sm.set_array([])  # We don't actually need data here
cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', pad=0.2, fraction=0.05)
cbar.set_label('Tumor Number (Sample_ID_j)')

# Show plot
plt.title("Heatmap of Shared Fraction with Tumor Number Intensity Ladder (X-axis)")
plt.show()


### 7.3 The number of samples a gRNA-BC appears

In [ ]:
# Count total unique Sample_IDs once (avoids recomputation)
total_unique_samples = input_df["Sample_ID"].nunique()

# Create a matrix indicating presence (1) of each gRNA_clonalbarcode in Sample_ID
presence_matrix = input_df.pivot_table(index="gRNA_clonalbarcode", columns="Sample_ID", 
                                       values="Cell_number", aggfunc="count").notna().astype(int)

# Compute unique Sample_ID count per gRNA_clonalbarcode (row-wise sum)
num_Sample_IDs = presence_matrix.sum(axis=1)

# Compute fraction of Sample_IDs per gRNA_clonalbarcode
fraction_Sample_IDs = num_Sample_IDs / total_unique_samples

# Construct output DataFrame
output_df = pd.DataFrame({
    "gRNA_clonalbarcode": num_Sample_IDs.index,
    "num_samples": num_Sample_IDs.values,
    "fraction_samples": fraction_Sample_IDs.values
})

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create the figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Plot histogram using seaborn
sns.histplot(output_df["fraction_samples"], bins=100, kde=False, ax=ax)

# Customize labels and title
ax.set_xlabel("Fraction of samples that a focal gRNA-BC appears")
ax.set_yscale("log",base=10)
ax.set_ylabel("Frequency")
ax.set_title("Distribution of repeated appreance")

# Show the plot
plt.show()

In [ ]:
temp_total_n = len(output_df.gRNA_clonalbarcode.unique())
output_df_s = output_df.groupby(['fraction_samples','num_samples'],as_index=False).count()
output_df_s.columns = ['fraction_samples','num_samples','frequency']
output_df_s['probability'] = output_df_s['frequency']/temp_total_n
output_df_s['probability_log10'] = np.log10(output_df_s['probability'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create the figure and axis
fig, ax = plt.subplots(figsize=(8, 6))
ix = 'fraction_samples'
iy = 'probability_log10'

# Scatter plot for individual points
sns.scatterplot(x=ix, y=iy, ax=ax, data=output_df_s, alpha=0.5)

# Add a trend line (LOWESS smoothed)
sns.regplot(x=ix, y=iy, ax=ax, data=output_df_s, scatter=False, color='red', line_kws={'linewidth': 2})

# Customize log scale and labels
# ax.set_yscale("log", base=10)  # Log scale for y-axis
ax.set_xlabel("Fraction of samples that a focal gRNA-BC appears")
ax.set_ylabel("Probability (log10)")
ax.set_title("log of the probability mass function (PMF) of a \nPoisson distribution with a small lambda approximately linear \nfor small values of k (the number of occurrences)")

# Show the plot
plt.show()


### 7.4 Share tumor number plot

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression

# Filter the data as in your original code
query_df = shared_fraction_long_df[shared_fraction_long_df['Sample_ID_i'] != shared_fraction_long_df['Sample_ID_j']]

# Define a function to plot and annotate outliers
def annotate_outliers(data, **kwargs):
    ax = plt.gca()
    
    # Scatter plot and regression line
    sns.regplot(x="Tumor_number_j", y="shared_fraction", data=data, ax=ax, 
                scatter_kws={'s': 10}, line_kws={'color': 'red'})
    
    # Fit a regression model
    X = data["Tumor_number_j"].values.reshape(-1, 1)
    y = data["shared_fraction"].values
    reg = LinearRegression().fit(X, y)
    predictions = reg.predict(X)
    
    # Calculate residuals and standard deviation of residuals
    residuals = y - predictions
    std_residuals = np.std(residuals)
    
    # Identify points beyond confidence interval (±2 standard deviations)
    outliers = np.abs(residuals) > 2 * std_residuals
    outlier_points = data[outliers]
    
    # Annotate outliers with Sample_ID_i
    for _, row in outlier_points.iterrows():
        ax.annotate(row['Sample_ID_j'], (row['Tumor_number_j'], row['shared_fraction']), 
                    textcoords="offset points", xytext=(5, 5), ha='left', fontsize=8, color='blue')

# Set up the FacetGrid to create a separate plot for each Sample_ID_i
g = sns.FacetGrid(query_df, col="Sample_ID_i", col_wrap=4, height=4, sharex=False, sharey=False)

# Apply the custom function to each facet
g.map_dataframe(annotate_outliers)

# Adjust labels and titles
g.set_axis_labels("Tumor Number (Sample_ID_j)", "Shared Fraction")
g.set_titles("Sample_ID_i: {col_name}")
g.fig.suptitle("Regression of Tumor Number (j) vs. Shared Fraction for Each Sample_ID_i", y=1.05)

# Show plot
plt.show()


### 8.3 quadratic fit

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, mean_squared_error, r2_score
import numpy as np

# Prepare the data for regression
# Copy the dataframe to avoid modifying the original
query_df = shared_fraction_long_df.copy()

# Compute Shared_tumor
query_df['Shared_tummor'] = query_df['shared_fraction'] * query_df['Tumor_number_i']

# Remove self-pairs
query_df = query_df[query_df['Sample_ID_i'] != query_df['Sample_ID_j']]

# Create a unique sorted pair identifier without modifying actual columns
query_df["sorted_pair"] = query_df.apply(lambda row: "_".join(sorted([row["Sample_ID_i"], row["Sample_ID_j"]])), axis=1)

# Drop duplicates based on the sorted pair while keeping the first occurrence
query_df = query_df.drop_duplicates(subset=["sorted_pair"]).drop(columns=["sorted_pair"])

# Remove missing values and reset the index
df = query_df.dropna().reset_index()

# Define independent variables (Tumor_number_i, Tumor_number_j) and target (Shared_tummor)
X = df[['Tumor_number_j', 'Tumor_number_i']].copy()  # Make a copy to avoid SettingWithCopyWarning
y = df['Shared_tummor']

# Add interaction term (Tumor_number_j * Tumor_number_i)
X['Interaction'] = X['Tumor_number_j'] * X['Tumor_number_i']


# Fit the multiple linear regression model with interaction term
model = LinearRegression()
model.fit(X, y)

# Get predictions and residuals
predicted = model.predict(X)
residuals = y - predicted

# Determine threshold for outliers (e.g., 3 standard deviations from the mean residual)
threshold = 3 * np.std(residuals)

# Identify outliers
outliers = df[np.abs(residuals) > threshold].copy()
outliers['Predicted_shared_tumor'] = predicted[outliers.index]

# Print regression coefficients for interpretation
print("Regression Coefficients:")
print(f"Tumor_number_j coefficient: {model.coef_[0]:.4f}")
print(f"Tumor_number_i coefficient: {model.coef_[1]:.4f}")
print(f"Interaction term coefficient: {model.coef_[2]:.4f}")
print(f"Intercept: {model.intercept_:.4f}")

# Scatter plot: Predicted vs Actual Shared Tumor Number
plt.figure(figsize=(5, 5))
plt.scatter(predicted, y, label="Data Points", alpha=0.6)
plt.plot(predicted, predicted, color="red", label="Perfect Fit Line")

# Label outliers with Sample_ID pairs (i, j)
for _, row in outliers.iterrows():
    ti = row["Sample_ID_i"]
    tj = row["Sample_ID_j"]
    plt.text(predicted[row.name], row["Shared_tummor"], f"({ti},{tj})", fontsize=10, color="red", ha="right")

# Plot settings
plt.xlabel("Predicted Shared Tumor Number")
plt.ylabel("Actual Shared Tumor Number")
plt.title("Regression with Interaction Term: \nPredicted vs Actual Shared Tumor Number")
plt.legend()
# plt.savefig(temp_out_address4_2)
plt.show()

# Define scoring functions for cross-validation
scoring = {
    'MSE': make_scorer(mean_squared_error, greater_is_better=False),
    'R2': make_scorer(r2_score)
}

# Perform 10-fold cross-validation
cv_results = cross_validate(model, X, y, scoring=scoring, cv=10)

# Calculate mean and standard deviation of cross-validation MSE and R^2 scores
mean_mse = -cv_results['test_MSE'].mean()  # Convert negative MSE back to positive
std_mse = cv_results['test_MSE'].std()
mean_r2 = cv_results['test_R2'].mean()
std_r2 = cv_results['test_R2'].std()

# Print cross-validation results
print(f"10-Fold Cross-Validation MSE: {mean_mse:.4f} ± {std_mse:.4f}")
print(f"10-Fold Cross-Validation R^2: {mean_r2:.4f} ± {std_r2:.4f}")


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# Copy the dataframe to avoid modifying the original
query_df = shared_fraction_long_df.copy()

# Compute Shared_tumor
query_df['Shared_tummor'] = query_df['shared_fraction'] * query_df['Tumor_number_i']

# Remove self-pairs
query_df = query_df[query_df['Sample_ID_i'] != query_df['Sample_ID_j']]

# Create a unique sorted pair identifier without modifying actual columns
query_df["sorted_pair"] = query_df.apply(lambda row: "_".join(sorted([row["Sample_ID_i"], row["Sample_ID_j"]])), axis=1)

# Drop duplicates based on the sorted pair while keeping the first occurrence
query_df = query_df.drop_duplicates(subset=["sorted_pair"]).drop(columns=["sorted_pair"])

# Remove missing values and reset the index
df = query_df.dropna().reset_index()

# Define independent variables (Tumor_number_i, Tumor_number_j) and target (Shared_tummor)
X = df[['Tumor_number_j', 'Tumor_number_i']].copy()  # Make a copy to avoid SettingWithCopyWarning
y = df['Shared_tummor']

# Add interaction term (Tumor_number_j * Tumor_number_i)
X['Interaction'] = X['Tumor_number_j'] * X['Tumor_number_i']

# Fit the linear Regressor model
model = LinearRegression()
model.fit(X, y)

# Get predictions and residuals
predicted = model.predict(X)
residuals = y - predicted

df['residual'] = residuals

# Calculate mean and standard deviation of residuals
mean_residual = df['residual'].mean()
std_residual = df['residual'].std()

# Define the threshold for filtering (2 standard deviations from the predicted line)
threshold = 2 * std_residual

# Filter DataFrame for positive residuals more than 2 standard deviations away from the predicted line
filtered_df_positive = df[df['residual'] > threshold]

# Initialize an undirected graph
G = nx.Graph()

# Add edges with weights for pairs that meet the positive threshold
for _, row in filtered_df_positive.iterrows():
    G.add_edge(row['Sample_ID_i'], row['Sample_ID_j'], weight=row['residual'], color='blue')

# Plot the undirected network graph
plt.figure(figsize=(10, 10))
pos = nx.spring_layout(G)  # Use spring layout for visual appeal

# Normalize edge weights for visualization
max_weight = max([data['weight'] for _, _, data in G.edges(data=True)]) if len(G.edges()) > 0 else 1
weights = [G[u][v]['weight'] / max_weight * 10 for u, v in G.edges()]  # Scale weights to a range (0-10)

# Draw nodes with size based on degree
node_sizes = [G.degree(node) * 100 for node in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color="skyblue", node_size=node_sizes)

# Draw undirected edges with color for positive residuals
edge_colors = [G[u][v]['color'] for u, v in G.edges()]
nx.draw_networkx_edges(
    G, pos, edgelist=G.edges(), width=weights, alpha=0.7, edge_color=edge_colors
)

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=10, font_color="black")

# Title and show plot
plt.title("Undirected Network of Shared gRNA_clonalbarcode Positive Residuals (More Than 2 SD Away from Predicted Line)")
plt.show()

In [ ]:
### 8.5 Examples 1

In [ ]:
test_sample = ['DL04_2','DL04_7']
query_df = input_df[input_df.Sample_ID.isin(test_sample)].copy()
query_df['Log10CN'] = np.log10(query_df['Cell_number'])
temp = pd.pivot_table(query_df, values = 'Log10CN', index=['gRNA_clonalbarcode','gRNA'], columns = 'Sample_ID').reset_index()
re_shaped_df = temp[~temp.isnull().any(axis=1)]

In [ ]:
df1 = query_df[query_df.Sample_ID==test_sample[0]].copy()
df2 = query_df[query_df.Sample_ID==test_sample[1]].copy()
df1['Kind'] = 'Unique'
df1.loc[df1.gRNA_clonalbarcode.isin(df2.gRNA_clonalbarcode),'Kind']='Shared'

df2['Kind'] = 'Unique'
df2.loc[df2.gRNA_clonalbarcode.isin(df1.gRNA_clonalbarcode),'Kind']='Shared'

In [ ]:
temp_df = df1.copy()
ax = sns.displot(data=temp_df, x="Log10CN", kind="hist", bins = 20,hue = 'Kind',
                 common_norm = False, aspect = 1.5,log_scale = [False,False],stat='probability',multiple = 'dodge' )

In [ ]:
temp_df = df2.copy()
ax = sns.displot(data=temp_df, x="Log10CN", kind="hist", bins = 20,hue = 'Kind',
                 common_norm = False, aspect = 1.5,log_scale = [False,False],stat='probability',multiple = 'dodge' )

In [ ]:
gs = gridspec.GridSpec(5, 17) 
fig1 = plt.figure(figsize=(17,5))
ax1=fig1.add_subplot(gs[:5, 0:5])
temp_df = re_shaped_df
ix = test_sample[0]
iy = test_sample[1]
sns.scatterplot(x=ix, y=iy, data=temp_df,ax= ax1)
ax1.set_xlabel(f'Tumor size for {ix}(log10)')
ax1.set_ylabel(f'Tumor size for {iy}(log10)')
temp1 = max(ax1.get_xlim()[0],ax1.get_ylim()[0])
temp2 = min(ax1.get_xlim()[1],ax1.get_ylim()[1])
diag_line, = ax1.plot((temp1,temp2),(temp1,temp2), ls="--", c=".3")
temp1 = scipy.stats.pearsonr(temp_df[ix],temp_df[iy])[0]
temp2 = scipy.stats.pearsonr(temp_df[ix],temp_df[iy])[1]
ax1.text(0.40,0.9, "Pearson's r = "+str(round(temp1,3)), size=10, ha="left",verticalalignment='center', transform=ax1.transAxes)
ax1.text(0.40,0.85, f"P-value = {temp2:.2f}", size=10, ha="left",verticalalignment='center', transform=ax1.transAxes)

 <font size="8" color='red'> All good</font>

## 7 Final sample to exclude 

<font size="8" color='red'> I drop nothing</font>

In [ ]:
sample_to_throw

## 8 Output data for bootstrapping analysis

In [ ]:
sample_to_throw = []

In [ ]:
Final_data = annotated_df[annotated_df.Identity=='gRNA'].copy()
Final_data = Final_data[~Final_data.Sample_ID.isin(sample_to_throw)]
Final_data['Type'] = 'Experiment'
sgInert_list_GW = pd.read_csv('data/sgInert.csv')['Gene']
Final_data.loc[Final_data.Targeted_gene_name.isin(sgInert_list_GW),'Type'] = 'Inert'
Final_data.to_parquet(final_data_output_address,index =False)

### 5.4 Combine with KT

In [ ]:
KT_df = pd.read_parquet('data/KT_reference_tumor.parquet')
KT_df = KT_df[KT_df.Mouse_genotype=='KT']

In [ ]:
merged_data_df = pd.concat([KT_df,Final_data],ignore_index=True)

In [ ]:
merged_data_df.to_parquet('data/GS_BTHC_KT_tumor.parquet',index=False)